
# Movie Ratings Analysis — Portfolio Project

**Objective:** Identify factors influencing movie ratings using exploratory data analysis and statistical modeling.

**Key Questions:**
- Do ratings change over time?
- Does budget influence ratings?
- Are some genres consistently better rated?


In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm

sns.set(style="whitegrid")


## 1. Data Loading

In [ ]:

# Replace with actual path
df = pd.read_csv("movies.csv")

df.head()


## 2. Data Cleaning

In [ ]:

# Rename columns for clarity
df = df.rename(columns={
    "date_x": "release_date",
    "budget_x": "budget"
})

# Convert dates
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year"] = df["release_date"].dt.year

# Remove invalid scores
df = df[df["score"] > 0].copy()

# Missing values analysis
missing_ratio = df.isna().mean().sort_values(ascending=False)
print(missing_ratio.head(10))

# Drop critical missing values
initial_size = len(df)
df = df.dropna(subset=["score", "genre", "budget", "year"])
print(f"Removed {(1 - len(df)/initial_size)*100:.2f}% of data")



## 3. Score Distribution

**Question:** How are movie ratings distributed?


In [ ]:

sns.histplot(df["score"], bins=30, kde=True)
plt.title("Distribution of Movie Ratings")
plt.show()



## 4. Ratings Over Time

**Question:** Have ratings changed over time?


In [ ]:

agg = df.groupby("year").agg(
    avg_score=("score", "mean"),
    count=("score", "count")
)

agg = agg[agg["count"] > 20]

plt.plot(agg.index, agg["avg_score"])
plt.title("Average Rating Over Time")
plt.xlabel("Year")
plt.ylabel("Average Score")
plt.show()



## 5. Budget vs Rating

**Question:** Do higher budgets lead to better ratings?


In [ ]:

df_clean = df[df["budget"] > 0].copy()
df_clean["log_budget"] = np.log(df_clean["budget"])

sns.scatterplot(data=df_clean, x="log_budget", y="score", alpha=0.3)
plt.title("Budget vs Rating")
plt.show()

corr = df_clean["log_budget"].corr(df_clean["score"])
print("Correlation:", corr)


### Regression Model

In [ ]:

X = sm.add_constant(df_clean["log_budget"])
y = df_clean["score"]

model = sm.OLS(y, X).fit()
print(model.summary())



## 6. Genre Analysis

**Question:** Are some genres better rated than others?


In [ ]:

top_genres = df["genre"].value_counts().head(10).index
df_genre = df[df["genre"].isin(top_genres)]

sns.boxplot(data=df_genre, x="genre", y="score")
plt.xticks(rotation=45)
plt.title("Ratings by Genre")
plt.show()

genre_stats = df.groupby("genre")["score"].agg(["mean","std","count"]).sort_values(by="mean", ascending=False)
genre_stats.head()



## 7. Multivariable Exploration


In [ ]:

sns.scatterplot(data=df_clean,
                x="log_budget",
                y="score",
                hue="genre",
                alpha=0.5)
plt.show()


## 8. Feature Engineering

In [ ]:

df["decade"] = (df["year"] // 10) * 10

sns.boxplot(data=df, x="decade", y="score")
plt.title("Ratings by Decade")
plt.show()



## 9. Conclusions

- Ratings remain relatively stable over time.
- Budget shows weak correlation with rating.
- Genre has a stronger effect than budget.
- Variability within genres is significant.

## 10. Next Steps

- Include director/actor features
- Build predictive models
- Perform cross-validation
